Project Name :Spend_DNA

Name : Mayuri Nilange

Batch : DS_DA_Oct2026


In [88]:
df = pd.read_csv("rahul_transactions.csv")
print("Dataset Loaded Successfully!")
print("Original Shape:", df.shape)
print("\nFirst 5 Transactions:")
print(df.head())
print("\nColumn Names:")
print(df.columns.tolist())

Dataset Loaded Successfully!
Original Shape: (1328, 8)

First 5 Transactions:
          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode        Ref  
0  678275.0   UPI  TXN190872  
1  681007.0   UPI  TXN143064  
2  484728.0  NEFT  TXN246316  
3 -748745.0   UPI  TXN569226  
4  680737.0   UPI  TXN968962  

Column Names:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']


In [89]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())
print("\nOriginal Transaction Types:")
print(df["Type"].value_counts(dropna=False))
print("\nDate Samples:")
print(df["Date"].head(10))
print("\nAmount Samples:")
print(df["Amount"].head(10))
print("\nMode Values:")
print(df["Mode"].value_counts(dropna=False))

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1328 non-null   object 
 1   Time         1328 non-null   object 
 2   Description  1328 non-null   object 
 3   Type         1328 non-null   object 
 4   Amount       1328 non-null   object 
 5   Balance      1328 non-null   float64
 6   Mode         1328 non-null   object 
 7   Ref          1328 non-null   object 
dtypes: float64(1), object(7)
memory usage: 83.1+ KB

Missing Values:
Date           0
Time           0
Description    0
Type           0
Amount         0
Balance        0
Mode           0
Ref            0
dtype: int64

Original Transaction Types:
Type
DR       670
Debit    652
CR         6
Name: count, dtype: int64

Date Samples:
0     2024-01-01
1      01-Jan-24
2      01-Jan-24
3     2024-01-01
4    01 Jan 2024
5     2024-01-01
6     2024-01-01
7

In [90]:
# Create a working copy
df_clean = df.copy()

# Remove exact duplicate transactions
original_count = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

duplicates_removed = original_count - len(df_clean)

# Parse mixed date formats
df_clean["date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

# Clean amount values
df_clean["amount"] = (
    df_clean["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_clean["amount"] = pd.to_numeric(
    df_clean["amount"],
    errors="coerce"
)

# Standardise transaction types
df_clean["type"] = (
    df_clean["Type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df_clean["type"] = df_clean["type"].replace({
    "dr": "debit",
    "cr": "credit"
})

# Treat empty Mode values as missing
df_clean["Mode"] = df_clean["Mode"].replace("", pd.NA)

# Count invalid dates and amounts
invalid_dates = df_clean["date"].isna().sum()
invalid_amounts = df_clean["amount"].isna().sum()

# Remove rows with invalid dates or amounts
df_clean = df_clean.dropna(
    subset=["date", "amount"]
).copy()

# Keep only standardised transaction types
df_clean = df_clean[
    df_clean["type"].isin(["debit", "credit"])
].copy()

# Reset index
df_clean = df_clean.reset_index(drop=True)

print("TRANSACTION PARSER COMPLETED")
print("=" * 55)

print("Original Transactions:", original_count)
print("Duplicates Removed:", duplicates_removed)
print("Invalid Dates:", invalid_dates)
print("Invalid Amounts:", invalid_amounts)
print("Final Transactions:", len(df_clean))

print("\nCleaned Dataset:")
print(df_clean.head())

print("\nData Types:")
print(df_clean.dtypes)

print("\nFinal Shape:", df_clean.shape)

TRANSACTION PARSER COMPLETED
Original Transactions: 1328
Duplicates Removed: 18
Invalid Dates: 0
Invalid Amounts: 0
Final Transactions: 1310

Cleaned Dataset:
          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode        Ref       date   amount    type  
0  678275.0   UPI  TXN190872 2024-01-01   2462.0   debit  
1  681007.0   UPI  TXN143064 2024-01-01     50.0   debit  
2  484728.0  NEFT  TXN246316 2024-01-01  84728.0  credit  
3 -748745.0   UPI  TXN569226 2024-01-01   1828.0   debit  
4  680737.0   UPI  TXN968962 2024-01-01    270.0   debit  

Data Types:
Date                   object

In [91]:
print("UNIQUE TRANSACTION DESCRIPTIONS")
print("=" * 55)
descriptions = df_clean["Description"].dropna().unique()
for i, description in enumerate(descriptions, start=1):
    print(i, ":", description)
print("\nTotal Unique Descriptions:", len(descriptions))

UNIQUE TRANSACTION DESCRIPTIONS
1 : AMAZON SELLER SVCS
2 : BHIM-BMTC
3 : NEFT-TECHCRUSH LABS-SALARY MAY24
4 : UPI-AMAN-8934@OKAXIS
5 : BHIM-BLINKIT
6 : BHIM ZEPTO
7 : UPI-UBER-2426@HDFCBANK
8 : POS SWIGGY BANGALORE
9 : UPI-GROWWPAY@HDFCBANK
10 : OLA ELECTRIC
11 : BMS MOVIE TICKETS
12 : POS OLA-PRIME
13 : SWIGGY-INSTAMART
14 : UPI-STARBUCKS@AXIS
15 : UPI-THIRDWAVE@OKAXIS
16 : ANI Technologies
17 : BMTC BUS PASS
18 : POS TRUFFLES
19 : FLIPKART INDIA
20 : POS SWIGGY-RESTAURANT
21 : GROFERS INDIA P L
22 : POS UBER BANGALORE
23 : BANGALORE ELEC SUPPLY
24 : TWC INDIA
25 : UPI-BESCOM-BILL@HDFCBANK
26 : UPI-AMAN-0816@OKAXIS
27 : ROPPEN TRANSPORTATION
28 : OLA CABS
29 : POS ZOMATO
30 : UPI-AMAZONPAY@HDFCBANK
31 : POS BLINKIT
32 : IMPS-RENT-LANDLORD-75500265
33 : ZOMATO MEDIA P L
34 : UPI-ANKIT-6430@OKAXIS
35 : UPI-OLACABS@HDFCBANK
36 : UPI-JIORECHARGE@PAYTM
37 : UPI-CCD@HDFCBANK
38 : Swiggy*Order
39 : INSTAMART BANGALORE
40 : UPI-ZOMATO-LIMITED@PAYTM
41 : AVENUE SUPERMARTS
42 : POS HP PETROL ST

In [92]:
vendor_dictionary = {

    "Swiggy": [
        "SWIGGY", "BUNDL"
    ],

    "Zomato": [
        "ZOMATO"
    ],

    "Amazon": [
        "AMAZON", "AMZN"
    ],

    "Zepto": [
        "ZEPTO"
    ],

    "Blinkit": [
        "BLINKIT", "GROFERS"
    ],

    "BigBasket": [
        "BIGBASKET", "BBNOW"
    ],

    "Dunzo": [
        "DUNZO"
    ],

    "Flipkart": [
        "FLIPKART"
    ],

    "Myntra": [
        "MYNTRA"
    ],

    "Ajio": [
        "AJIO"
    ],

    "Nykaa": [
        "NYKAA"
    ],

    "Uber": [
        "UBER"
    ],

    "Ola": [
        "OLA"
    ],

    "Rapido": [
        "RAPIDO"
    ],

    "BMTC": [
        "BMTC"
    ],

    "Namma Metro": [
        "NAMMA METRO", "BMRCL"
    ],

    "Starbucks": [
        "STARBUCKS"
    ],

    "Cafe Coffee Day": [
        "CAFE COFFEE DAY", "CCD"
    ],

    "Third Wave Coffee": [
        "THIRD WAVE"
    ],

    "McDonalds": [
        "MCDONALD", "MCD"
    ],

    "Dominos": [
        "DOMINO"
    ],
    "Pizza Hut": [
        "PIZZA HUT"
    ],
    "KFC": [
        "KFC"
    ],
    "Subway": [
        "SUBWAY"
    ],
    "DMart": [
        "DMART"
    ],
    "Reliance Fresh": [
        "RELIANCE FRESH"
    ],
    "Netflix": [
        "NETFLIX"
    ],
    "Spotify": [
        "SPOTIFY"
    ],
    "Amazon Prime": [
        "PRIME VIDEO"
    ],
    "Jio": [
        "JIO"
    ],
    "Airtel": [
        "AIRTEL"
    ],
    "BESCOM": [
        "BESCOM"
    ],
    "Zerodha": [
        "ZERODHA"
    ],
    "Groww": [
        "GROWW"
    ],

    "Indian Oil": [
        "INDIAN OIL", "IOCL"
    ],
    "HPCL": [
        "HPCL"
    ],
    "BookMyShow": [
        "BOOKMYSHOW"
    ]
}
print("Vendor Dictionary Created!")
print("Total Canonical Vendors:", len(vendor_dictionary))

Vendor Dictionary Created!
Total Canonical Vendors: 37


In [93]:
def extract_vendor(description):

    if pd.isna(description):
        return "Uncategorised"

    description = str(description).upper()

    # Special case: ATM withdrawals
    if "ATM-WDL" in description or "ATM WDL" in description:
        return "Cash Withdrawal"

    # Check known merchants
    for vendor, keywords in vendor_dictionary.items():

        for keyword in keywords:

            if keyword in description:
                return vendor

    # Identify person-to-person UPI transfers
    if "UPI-" in description and "@" in description:
        return "P2P Transfer"

    if "BHIM " in description and "@" in description:
        return "P2P Transfer"

    return "Uncategorised"


df_clean["vendor_clean"] = df_clean["Description"].apply(
    extract_vendor
)

print("VENDOR EXTRACTION COMPLETED")
print("=" * 55)

print("\nUnique Canonical Vendors:")
print(df_clean["vendor_clean"].nunique())

print("\nTop 10 Vendors:")
print(
    df_clean["vendor_clean"]
    .value_counts()
    .head(10)
)

print("\nUncategorised Descriptions:")
print(
    df_clean.loc[
        df_clean["vendor_clean"] == "Uncategorised",
        "Description"
    ].unique()
)

VENDOR EXTRACTION COMPLETED

Unique Canonical Vendors:
29

Top 10 Vendors:
vendor_clean
Uncategorised    223
Swiggy           223
Zomato           121
Amazon            86
Uber              71
Ola               69
P2P Transfer      62
Zepto             58
Blinkit           55
Flipkart          43
Name: count, dtype: int64

Uncategorised Descriptions:
['NEFT-TECHCRUSH LABS-SALARY MAY24' 'BMS MOVIE TICKETS' 'ANI Technologies'
 'POS TRUFFLES' 'BANGALORE ELEC SUPPLY' 'TWC INDIA'
 'ROPPEN TRANSPORTATION' 'IMPS-RENT-LANDLORD-75500265'
 'INSTAMART BANGALORE' 'AVENUE SUPERMARTS' 'POS HP PETROL STATION'
 'POS BANGALORE RESTAURANT' 'VI POSTPAID' 'POS BPCL PETROL PUMP'
 'POS DINEOUT' 'COFFEE DAY GLOBAL' 'DISNEY HOTSTAR'
 'POS EMPIRE RESTAURANT' 'POS MEGHANA FOODS' 'FKART INTRNET'
 'IMPS-RENT-LANDLORD-39598076' 'FSN E-COMMERCE' 'BIGTREE ENTERTAINMENT'
 'INNOVATIVE RETAIL' 'POS INSTAMART' 'STAR INDIA PVT LTD'
 'KIRANAKART TECH' 'IMPS-RENT-LANDLORD-51148088' 'BWSSB WATER BILL'
 'IMPS-RENT-LANDLORD-3

In [94]:
category_dictionary = {

    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "Dunzo": "Quick Commerce",

    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Ajio": "E-commerce",
    "Nykaa": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",
    "Namma Metro": "Transport",

    "Starbucks": "Cafe",
    "Cafe Coffee Day": "Cafe",
    "Third Wave Coffee": "Cafe",

    "McDonalds": "Restaurants",
    "Dominos": "Restaurants",
    "Pizza Hut": "Restaurants",
    "KFC": "Restaurants",
    "Subway": "Restaurants",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Amazon Prime": "Subscriptions",

    "Jio": "Utilities",
    "Airtel": "Utilities",
    "BESCOM": "Utilities",

    "BigBasket": "Groceries",
    "DMart": "Groceries",
    "Reliance Fresh": "Groceries",

    "Zerodha": "Investments",
    "Groww": "Investments",

    "Indian Oil": "Fuel",
    "HPCL": "Fuel",

    "BookMyShow": "Entertainment",

    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal"
}

print("Category Dictionary Created!")
print("Total Category Mappings:", len(category_dictionary))

Category Dictionary Created!
Total Category Mappings: 39


In [95]:
df_clean["category"] = df_clean["vendor_clean"].map(
    category_dictionary
)

df_clean["category"] = df_clean["category"].fillna(
    "Uncategorised"
)

print("CATEGORY TAGGING COMPLETED")
print("=" * 55)

print("\nCategory Distribution:")

print(
    df_clean["category"].value_counts()
)

print("\nUnique Categories:")
print(df_clean["category"].nunique())

print("\nUncategorised Transactions:")

print(
    df_clean.loc[
        df_clean["category"] == "Uncategorised",
        ["Description", "vendor_clean"]
    ]
)

CATEGORY TAGGING COMPLETED

Category Distribution:
category
Food Delivery        344
Uncategorised        223
Transport            218
E-commerce           165
Quick Commerce       113
Cafe                  68
Personal Transfer     62
Investments           23
Groceries             23
Utilities             21
Subscriptions         18
Cash Withdrawal       17
Fuel                  11
Entertainment          4
Name: count, dtype: int64

Unique Categories:
14

Uncategorised Transactions:
                           Description   vendor_clean
2     NEFT-TECHCRUSH LABS-SALARY MAY24  Uncategorised
11                   BMS MOVIE TICKETS  Uncategorised
16                    ANI Technologies  Uncategorised
18                        POS TRUFFLES  Uncategorised
24               BANGALORE ELEC SUPPLY  Uncategorised
...                                ...            ...
1285                 VODAFONE IDEA LTD  Uncategorised
1287             POS HP PETROL STATION  Uncategorised
1292                  ANI 

In [96]:
# Separate debit and credit transactions

credits = df_clean[
    df_clean["type"] == "credit"
]

debits = df_clean[
    df_clean["type"] == "debit"
]

# Total credits and debits

total_credits = credits["amount"].sum()
total_debits = debits["amount"].sum()

# Net savings

net_savings = total_credits - total_debits

# Savings rate

if total_credits > 0:
    savings_rate = (net_savings / total_credits) * 100
else:
    savings_rate = 0

# Exclude personal transfers and cash withdrawals
# from consumption category calculations

spending = debits[
    ~debits["category"].isin([
        "Personal Transfer",
        "Cash Withdrawal",
        "Uncategorised"
    ])
].copy()

total_spending = spending["amount"].sum()

# Transaction count

total_transactions = len(df_clean)

# Unique vendors

unique_vendors = df_clean["vendor_clean"].nunique()

print("FINANCIAL SUMMARY")
print("=" * 55)

print(f"Total Credits       : Rs. {total_credits:,.2f}")
print(f"Total Debits        : Rs. {total_debits:,.2f}")
print(f"Net Savings         : Rs. {net_savings:,.2f}")
print(f"Savings Rate        : {savings_rate:.2f}%")
print(f"Total Transactions  : {total_transactions}")
print(f"Unique Vendors      : {unique_vendors}")
print(f"Consumption Spend   : Rs. {total_spending:,.2f}")

if net_savings < 0:
    print("\nFinancial Status: OVERSPENDING")
else:
    print("\nFinancial Status: SAVING")

FINANCIAL SUMMARY
Total Credits       : Rs. 509,774.00
Total Debits        : Rs. 1,678,901.00
Net Savings         : Rs. -1,169,127.00
Savings Rate        : -229.34%
Total Transactions  : 1310
Unique Vendors      : 29
Consumption Spend   : Rs. 1,229,991.00

Financial Status: OVERSPENDING


In [99]:
# Category-wise spending

category_spending = (
    spending.groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

# Category percentages

category_percent = (
    category_spending / total_spending
) * 100

print("TOP SPENDING CATEGORIES")
print("=" * 65)

for category, amount in category_spending.head(12).items():

    percentage = category_percent[category]

    bar_length = int(percentage / 2)

    bar = "#" * bar_length

    print(
        f"{category:<20} "
        f"{bar:<20} "
        f"{percentage:>6.2f}% "
        f"Rs. {amount:,.2f}"
    )


# Vendor-wise spending

vendor_spending = (
    spending.groupby("vendor_clean")["amount"]
    .agg(["sum", "count"])
    .sort_values("sum", ascending=False)
)

print("\nTOP 5 VENDORS")
print("=" * 65)

for vendor, row in vendor_spending.head(5).iterrows():

    print(
        f"{vendor:<20} "
        f"Rs. {row['sum']:>12,.2f} "
        f"({int(row['count'])} transactions)"
    )

TOP SPENDING CATEGORIES
E-commerce           ########################  48.20% Rs. 592,816.00
Investments          ##########            20.18% Rs. 248,160.00
Food Delivery        ######                12.26% Rs. 150,839.00
Quick Commerce       ##                     4.89% Rs. 60,152.00
Transport            ##                     4.09% Rs. 50,270.00
Fuel                 #                      3.36% Rs. 41,287.00
Groceries            #                      2.72% Rs. 33,398.00
Utilities                                   1.84% Rs. 22,657.00
Cafe                                        1.81% Rs. 22,216.00
Subscriptions                               0.45% Rs. 5,504.00
Entertainment                               0.22% Rs. 2,692.00

TOP 5 VENDORS
Amazon               Rs.   328,530.00 (86 transactions)
Zerodha              Rs.   210,000.00 (14 transactions)
Flipkart             Rs.   170,745.00 (43 transactions)
Swiggy               Rs.    95,523.00 (223 transactions)
Myntra               Rs.   

In [104]:
# DAY 4 - CELL 12
# Monthly Spending Analysis

from datetime import datetime

# Extract month number
spending["month"] = spending["date"].dt.month

# Create monthly category matrix
month_pivot = spending.pivot_table(
    values="amount",
    index="category",
    columns="month",
    aggfunc="sum",
    fill_value=0
)

# Ensure all six months are present
month_pivot = month_pivot.reindex(
    columns=[1, 2, 3, 4, 5, 6],
    fill_value=0
)

# Rename month columns
month_pivot.columns = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun"
]

print("MONTHLY CATEGORY SPENDING")
print("=" * 75)
print(month_pivot.round(2))

# Total monthly spending
monthly_total = spending.groupby("month")["amount"].sum()

print("\nTOTAL MONTHLY SPENDING")
print("=" * 55)

for month, amount in monthly_total.items():

    month_name = datetime(
        2024, int(month), 1
    ).strftime("%B")

    print(f"{month_name:<15} Rs. {amount:,.2f}")



MONTHLY CATEGORY SPENDING
                    Jan      Feb       Mar      Apr      May       Jun
category                                                              
Cafe             1774.0   3223.0    3900.0   4745.0   4534.0    4040.0
E-commerce      96941.0  91018.0  105977.0  65856.0  95776.0  137248.0
Entertainment     952.0      0.0     562.0   1178.0      0.0       0.0
Food Delivery   22633.0  23740.0   24803.0  27756.0  25408.0   26499.0
Fuel            11084.0   1452.0   11942.0   7828.0   7188.0    1793.0
Groceries       11763.0   7517.0    1865.0   4916.0   5888.0    1449.0
Investments     38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Quick Commerce   9995.0  12459.0   12229.0   8221.0  10139.0    7109.0
Subscriptions     453.0   1036.0    1598.0    538.0    538.0    1341.0
Transport       10436.0   9797.0    5667.0   7399.0  11359.0    5612.0
Utilities        3272.0   6870.0    3517.0   5177.0   3092.0     729.0

TOTAL MONTHLY SPENDING
January         Rs. 207,735

In [103]:
# Monthly Growth and Decline Analysis
monthly_changes = []

for category in month_pivot.index:

    values = np.array(
        month_pivot.loc[category].values,
        dtype=float
    )

    for i in range(1, len(values)):

        previous_month = values[i - 1]
        current_month = values[i]

        if previous_month > 0:

            change = (
                (current_month - previous_month)
                / previous_month
            ) * 100

            monthly_changes.append({
                "category": category,
                "month": month_pivot.columns[i],
                "change": change
            })

print("MONTHLY GROWTH ANALYSIS")
print("=" * 60)

if monthly_changes:

    biggest_growth = max(
        monthly_changes,
        key=lambda x: x["change"]
    )

    biggest_decline = min(
        monthly_changes,
        key=lambda x: x["change"]
    )

    print("\nBIGGEST MONTHLY GROWTH")

    print("Category:", biggest_growth["category"])
    print("Month:", biggest_growth["month"])
    print(f"Growth: {biggest_growth['change']:.2f}%")

    print("\nBIGGEST MONTHLY DECLINE")

    print("Category:", biggest_decline["category"])
    print("Month:", biggest_decline["month"])
    print(f"Change: {biggest_decline['change']:.2f}%")

else:
    print("No monthly changes available.")



MONTHLY GROWTH ANALYSIS

BIGGEST MONTHLY GROWTH
Category: Fuel
Month: Mar
Growth: 722.45%

BIGGEST MONTHLY DECLINE
Category: Entertainment
Month: Feb
Change: -100.00%


In [105]:
# Extract hour from transaction time
spending["hour"] = pd.to_numeric(
    spending["Time"].astype(str).str[:2],
    errors="coerce"
)

# Remove invalid hours
spending = spending.dropna(subset=["hour"]).copy()

spending["hour"] = spending["hour"].astype(int)

# Keep only valid hours
spending = spending[
    spending["hour"].between(0, 23)
].copy()

# Create category-hour spending matrix
hour_pivot = spending.pivot_table(
    values="amount",
    index="category",
    columns="hour",
    aggfunc="sum",
    fill_value=0
)

# Include all 24 hours
hour_pivot = hour_pivot.reindex(
    columns=range(24),
    fill_value=0
)

print("CATEGORY-WISE HOURLY SPENDING")
print("=" * 75)

print(hour_pivot.round(2))

# ASCII heatmap
print("\nTEXT-BASED SPENDING HEATMAP")
print("=" * 75)

for category in hour_pivot.index:

    values = hour_pivot.loc[category].values
    maximum = max(values)

    symbols = ""

    for value in values:

        if value == 0:
            symbols += " "

        elif maximum > 0 and value / maximum >= 0.75:
            symbols += "#"

        elif maximum > 0 and value / maximum >= 0.50:
            symbols += "*"

        elif maximum > 0 and value / maximum >= 0.25:
            symbols += ":"

        else:
            symbols += "."

    print(f"{category:<20} |{symbols}|")

print("\nHours: 00 01 02 03 04 05 06 07 08 09 10 11")
print("       12 13 14 15 16 17 18 19 20 21 22 23")

print("\nLegend:")
print("# = Very High")
print("* = High")
print(": = Medium")
print(". = Low")
print("Blank = No Spending")



CATEGORY-WISE HOURLY SPENDING
hour                 0        1        2        3        4        5        6   \
category                                                                        
Cafe              309.0    201.0      0.0    316.0      0.0      0.0    179.0   
E-commerce      18650.0   9016.0  10246.0  14865.0  12018.0  17258.0   7866.0   
Entertainment     562.0      0.0      0.0      0.0      0.0      0.0      0.0   
Food Delivery     832.0   3017.0    928.0   1702.0   2618.0   3488.0    238.0   
Fuel             2675.0      0.0      0.0    676.0   2010.0   2105.0      0.0   
Groceries           0.0   1536.0    635.0   7080.0   1169.0      0.0   3645.0   
Investments      4883.0  15000.0      0.0      0.0  18834.0   4496.0  37717.0   
Quick Commerce    859.0   1402.0    697.0    732.0    768.0   1448.0    787.0   
Subscriptions       0.0      0.0    119.0    669.0   1998.0    745.0    657.0   
Transport         646.0    260.0    358.0    337.0    577.0   1313.0    555.0  

In [106]:
# Define late-night transactions
late_night = spending[
    (spending["hour"] >= 21) |
    (spending["hour"] < 2)
]

# Identify food delivery transactions
food_delivery = spending[
    spending["category"] == "Food Delivery"
]

# Identify late-night food delivery
food_late_night = food_delivery[
    (food_delivery["hour"] >= 21) |
    (food_delivery["hour"] < 2)
]

# Calculate late-night food delivery percentage
if len(food_delivery) > 0:

    late_night_percentage = (
        len(food_late_night) /
        len(food_delivery)
    ) * 100

else:
    late_night_percentage = 0

# Calculate late-night spending amount
late_night_amount = late_night["amount"].sum()

print("SPENDDNA - TIME-OF-DAY INSIGHTS")
print("=" * 65)

print(
    f"\nTotal Late-Night Transactions: "
    f"{len(late_night)}"
)

print(
    f"Late-Night Spending: "
    f"Rs. {late_night_amount:,.2f}"
)

print(
    f"Food Delivery Transactions: "
    f"{len(food_delivery)}"
)

print(
    f"Late-Night Food Orders: "
    f"{len(food_late_night)}"
)

print(
    f"Late-Night Food Percentage: "
    f"{late_night_percentage:.2f}%"
)

# Find peak spending hour for every category
print("\nPEAK SPENDING HOURS")
print("=" * 65)

for category in hour_pivot.index:

    values = hour_pivot.loc[category]

    peak_hour = values.idxmax()
    peak_amount = values.max()

    print(
        f"{category:<20} "
        f"{peak_hour:02d}:00 "
        f"Rs. {peak_amount:,.2f}"
    )

# Identify spending personality pattern
print("\nLATE-NIGHT SPENDING BEHAVIOUR")
print("=" * 65)

if late_night_percentage > 50:

    print("Rahul shows a strong late-night food ordering habit.")

elif late_night_percentage > 25:

    print("Rahul shows a moderate late-night food ordering habit.")

else:

    print("Rahul shows a low late-night food ordering habit.")



SPENDDNA - TIME-OF-DAY INSIGHTS

Total Late-Night Transactions: 175
Late-Night Spending: Rs. 219,585.00
Food Delivery Transactions: 344
Late-Night Food Orders: 71
Late-Night Food Percentage: 20.64%

PEAK SPENDING HOURS
Cafe                 10:00 Rs. 3,257.00
E-commerce           14:00 Rs. 72,584.00
Entertainment        20:00 Rs. 952.00
Food Delivery        20:00 Rs. 18,372.00
Fuel                 21:00 Rs. 11,845.00
Groceries            03:00 Rs. 7,080.00
Investments          10:00 Rs. 90,000.00
Quick Commerce       19:00 Rs. 6,992.00
Subscriptions        04:00 Rs. 1,998.00
Transport            20:00 Rs. 4,260.00
Utilities            08:00 Rs. 4,325.00

LATE-NIGHT SPENDING BEHAVIOUR
Rahul shows a low late-night food ordering habit.


In [107]:
# Anomaly Detection
debit_data = df_clean[df_clean["type"] == "debit"].copy()

# Calculate category-wise average and standard deviation
debit_data["category_mean"] = (
    debit_data.groupby("category")["amount"].transform("mean")
)

debit_data["category_std"] = (
    debit_data.groupby("category")["amount"].transform("std")
)

# Calculate z-score
debit_data["z_score"] = (
    (debit_data["amount"] - debit_data["category_mean"])
    / debit_data["category_std"]
)

# Flag transactions with z-score greater than 2
anomalies = debit_data[
    debit_data["z_score"] > 2
].copy()

anomalies = anomalies.sort_values(
    by="z_score", ascending=False
)

print("ANOMALY DETECTION")
print("=" * 60)
print("Total debit transactions:", len(debit_data))
print("Total anomalies detected:", len(anomalies))

print("\nTOP 5 ANOMALOUS TRANSACTIONS")

print(
    anomalies[
        ["date", "vendor_clean", "category", "amount", "z_score"]
    ].head(5).to_string(index=False)
)

ANOMALY DETECTION
Total debit transactions: 1304
Total anomalies detected: 31

TOP 5 ANOMALOUS TRANSACTIONS
      date  vendor_clean          category  amount  z_score
2024-01-22  P2P Transfer Personal Transfer  7264.0 5.932391
2024-01-05 Uncategorised     Uncategorised 18000.0 5.252254
2024-02-05 Uncategorised     Uncategorised 18000.0 5.252254
2024-04-05 Uncategorised     Uncategorised 18000.0 5.252254
2024-03-03 Uncategorised     Uncategorised 18000.0 5.252254


In [108]:
# Prepare metrics for archetype detection
total_debit = debit_data["amount"].sum()
total_credit = df_clean[
    df_clean["type"] == "credit"
]["amount"].sum()

category_spend = debit_data.groupby("category")["amount"].sum()

category_percent = (
    category_spend / total_debit * 100
).to_dict()

# Extract transaction hour
debit_data["hour"] = pd.to_numeric(
    debit_data["Time"].astype(str).str[:2],
    errors="coerce"
)

# Keep valid hours only
debit_data = debit_data.dropna(subset=["hour"]).copy()
debit_data["hour"] = debit_data["hour"].astype(int)

# Food delivery transactions
food_orders = debit_data[
    debit_data["category"] == "Food Delivery"
].copy()

# Late night = 9 PM to before 2 AM
late_night_food = food_orders[
    (food_orders["hour"] >= 21) |
    (food_orders["hour"] < 2)
]

if len(food_orders) > 0:
    late_night_food_percent = (
        len(late_night_food) / len(food_orders)
    ) * 100
else:
    late_night_food_percent = 0

# Savings rate
if total_credit > 0:
    savings_rate = (
        (total_credit - total_debit) / total_credit
    ) * 100
else:
    savings_rate = 0

print("ARCHETYPE METRICS")
print("=" * 50)

for category, percentage in sorted(
    category_percent.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{category:<22}: {percentage:.2f}%")

print("\nLate-night food orders:", len(late_night_food))
print("Late-night food percentage:", round(late_night_food_percent, 2), "%")
print("Savings rate:", round(savings_rate, 2), "%")

ARCHETYPE METRICS
E-commerce            : 35.31%
Uncategorised         : 19.99%
Investments           : 14.78%
Food Delivery         : 8.98%
Personal Transfer     : 4.04%
Quick Commerce        : 3.58%
Transport             : 2.99%
Cash Withdrawal       : 2.71%
Fuel                  : 2.46%
Groceries             : 1.99%
Utilities             : 1.35%
Cafe                  : 1.32%
Subscriptions         : 0.33%
Entertainment         : 0.16%

Late-night food orders: 71
Late-night food percentage: 20.64 %
Savings rate: -229.34 %


In [110]:
# DAY 5 - CELL 18
# Spending Archetype Detection

archetypes = []

def detect_foodie():
    food_categories = [
        "Food Delivery",
        "Restaurants",
        "Cafe"
    ]

    food_percent = sum(
        category_percent.get(category, 0)
        for category in food_categories
    )

    if food_percent > 25:
        archetypes.append(
            ("THE FOODIE", f"{food_percent:.2f}% spent on food")
        )


def detect_quick_commerce_junkie():
    value = category_percent.get("Quick Commerce", 0)

    if value > 15:
        archetypes.append(
            ("THE QUICK COMMERCE JUNKIE", f"{value:.2f}% on Quick Commerce")
        )


def detect_shopaholic():
    value = category_percent.get("E-commerce", 0)

    if value > 15:
        archetypes.append(
            ("THE SHOPAHOLIC", f"{value:.2f}% on E-commerce")
        )


def detect_investor():
    value = category_percent.get("Investments", 0)

    if value > 15:
        archetypes.append(
            ("THE INVESTOR", f"{value:.2f}% on Investments")
        )


def detect_late_night_snacker():
    if late_night_food_percent > 50:
        archetypes.append(
            ("THE LATE-NIGHT SNACKER",
             f"{late_night_food_percent:.2f}% of food orders after 9 PM")
        )


def detect_cab_commuter():
    value = category_percent.get("Transport", 0)

    if value > 10:
        archetypes.append(
            ("THE CAB COMMUTER", f"{value:.2f}% on Transport")
        )


def detect_subscription_lover():
    subscription_vendors = debit_data[
        debit_data["category"] == "Subscriptions"
    ]["vendor_clean"].nunique()

    if subscription_vendors >= 5:
        archetypes.append(
            ("THE SUBSCRIPTION LOVER",
             f"{subscription_vendors} different subscription vendors")
        )


def detect_yolo_spender():
    if savings_rate < 10:
        archetypes.append(
            ("THE YOLO SPENDER", f"Savings rate: {savings_rate:.2f}%")
        )


def detect_disciplined_saver():
    if savings_rate > 40:
        archetypes.append(
            ("THE DISCIPLINED SAVER",
             f"Savings rate: {savings_rate:.2f}%")
        )


# Apply all detection rules
detect_foodie()
detect_quick_commerce_junkie()
detect_shopaholic()
detect_investor()
detect_late_night_snacker()
detect_cab_commuter()
detect_subscription_lover()
detect_yolo_spender()
detect_disciplined_saver()

print("RAHUL'S SPENDING ARCHETYPES")
print("=" * 60)

if archetypes:
    for name, reason in archetypes:
        print(f"-> {name:<30} ({reason})")
else:
    print("No archetypes detected.")

RAHUL'S SPENDING ARCHETYPES
-> THE SHOPAHOLIC                 (35.31% on E-commerce)
-> THE YOLO SPENDER               (Savings rate: -229.34%)


In [116]:
# Final Formatted Console Report
print("\n" + "=" * 75)
print("                 SpendDNA REPORT - RAHUL SHARMA")
print("=" * 75)
# Date range
start_date = df_clean["date"].min()
end_date = df_clean["date"].max()

print(
    f"  Period: {start_date.strftime('%d %b %Y')} "
    f"to {end_date.strftime('%d %b %Y')}"
)
print(f"  Total Transactions: {len(df_clean)}")
# EXECUTIVE SUMMARY
print("\n" + "-" * 75)
print("  1. EXECUTIVE SUMMARY")
print("-" * 75)

net_change = total_credit - total_debit

print(f"  Total Credits       : Rs. {total_credit:,.2f}")
print(f"  Total Debits        : Rs. {total_debit:,.2f}")
print(f"  Net Change          : Rs. {net_change:,.2f}")
print(f"  Savings Rate        : {savings_rate:.2f}%")
print(f"  Transactions        : {len(df_clean)}")
print(f"  Unique Vendors      : {df_clean['vendor_clean'].nunique()}")

if net_change < 0:
    print("  Financial Status    : OVERSPENDING")
else:
    print("  Financial Status    : POSITIVE SAVINGS")

# TOP CATEGORIES
print("\n" + "-" * 75)
print("  2. TOP SPENDING CATEGORIES")
print("-" * 75)

top_categories = category_spend.sort_values(
    ascending=False
).head(5)

for category, amount in top_categories.items():
    percentage = category_percent.get(category, 0)
    bar = "#" * int(percentage / 2)

    print(
        f"  {category:<20} {bar:<20} "
        f"{percentage:>6.2f}%  Rs. {amount:,.2f}"
    )

# TOP VENDORS
print("\n" + "-" * 75)
print("  3. TOP VENDORS")
print("-" * 75)

top_vendors = (
    debit_data.groupby("vendor_clean")["amount"]
    .agg(["sum", "count"])
    .sort_values("sum", ascending=False)
    .head(5)
)

for vendor, row in top_vendors.iterrows():
    print(
        f"  {vendor:<22} Rs. {row['sum']:>12,.2f} "
        f"({int(row['count'])} transactions)"
    )

# MONTHLY FOOD DELIVERY TREND
print("\n" + "-" * 75)
print("  4. MONTHLY FOOD DELIVERY TREND")
print("-" * 75)

food_delivery_data = debit_data[
    debit_data["category"] == "Food Delivery"
].copy()

monthly_report = food_delivery_data.groupby(
    food_delivery_data["date"].dt.month
)["amount"].sum()

if len(monthly_report) > 0:
    for month, amount in monthly_report.items():
        bar = "#" * int(
            amount / max(monthly_report.max(), 1) * 25
        )

        print(
            f"  {month_names[int(month) - 1]:<12} "
            f"Rs. {amount:>12,.2f}  {bar}"
        )
else:
    print("  No Food Delivery monthly data available.")

# TIME OF DAY
print("\n" + "-" * 75)
print("  5. TIME-OF-DAY PATTERNS")
print("-" * 75)

if len(food_orders) > 0:
    print(
        f"  Late-night Food Orders : "
        f"{late_night_food_percent:.2f}%"
    )

category_hour = debit_data.groupby(
    ["category", "hour"]
)["amount"].sum()

for category in ["Food Delivery", "Cafe", "Quick Commerce"]:
    if category in debit_data["category"].values:
        hours = category_hour.loc[category]
        peak_hour = int(hours.idxmax())

        print(
            f"  {category:<20}: Peak spending at {peak_hour:02d}:00"
        )

# ANOMALIES
print("\n" + "-" * 75)
print("  6. ANOMALY DETECTION")
print("-" * 75)

print("  Total Anomalies Found:", len(anomalies))
print("\n  Top 5 unusual transactions:")

for _, row in anomalies.head(5).iterrows():
    print(
        f"  {str(row['date'])[:10]} | "
        f"{row['vendor_clean']:<18} | "
        f"{row['category']:<18} | "
        f"Rs. {row['amount']:>10,.2f} | "
        f"Z = {row['z_score']:.2f}"
    )

# ARCHETYPES
print("\n" + "-" * 75)
print("  7. RAHUL'S SPENDING ARCHETYPES")
print("-" * 75)

for name, reason in archetypes:
    print(f"  -> {name:<30} {reason}")

# KEY INSIGHTS
print("\n" + "-" * 75)
print("  8. KEY FINANCIAL SUMMARY")
print("-" * 75)

if len(top_categories) > 0:
    print(
        f"  Highest spending category: "
        f"{top_categories.index[0]}"
    )

if len(top_vendors) > 0:
    print(f"  Most-used top vendor by spend: {top_vendors.index[0]}")

print(f"  Total unusual transactions: {len(anomalies)}")

print("\n" + "=" * 75)
print("               END OF SpendDNA REPORT")
print("=" * 75)


                 SpendDNA REPORT - RAHUL SHARMA
  Period: 01 Jan 2024 to 30 Jun 2024
  Total Transactions: 1310

---------------------------------------------------------------------------
  1. EXECUTIVE SUMMARY
---------------------------------------------------------------------------
  Total Credits       : Rs. 509,774.00
  Total Debits        : Rs. 1,678,901.00
  Net Change          : Rs. -1,169,127.00
  Savings Rate        : -229.34%
  Transactions        : 1310
  Unique Vendors      : 29
  Financial Status    : OVERSPENDING

---------------------------------------------------------------------------
  2. TOP SPENDING CATEGORIES
---------------------------------------------------------------------------
  E-commerce           #################     35.31%  Rs. 592,816.00
  Uncategorised        #########             19.99%  Rs. 335,585.00
  Investments          #######               14.78%  Rs. 248,160.00
  Food Delivery        ####                   8.98%  Rs. 150,839.00
  Persona

In [113]:
print("\n" + "=" * 65)
print("                 SpendDNA - KEY INSIGHTS")
print("=" * 65)

# Insight 1: Top spending category
if len(category_spend) > 0:
    highest_category = category_spend.idxmax()
    highest_amount = category_spend.max()
    highest_percent = category_percent.get(highest_category, 0)

    print("\n1. SPENDING HABIT")
    print(
        f"   {highest_category} is the highest spending category, "
        f"accounting for {highest_percent:.2f}% of total debits "
        f"(Rs. {highest_amount:,.2f})."
    )

# Insight 2: Late-night food behaviour
print("\n2. TIME-OF-DAY BEHAVIOUR")

if len(food_orders) > 0:
    print(
        f"   {late_night_food_percent:.2f}% of Food Delivery "
        "transactions happen between 9 PM and 2 AM."
    )
else:
    print("   No Food Delivery transactions were found.")

# Insight 3: Savings behaviour
print("\n3. SAVINGS BEHAVIOUR")

if savings_rate < 0:
    print(
        f"   Total debits exceed total credits. "
        f"The savings rate is {savings_rate:.2f}%, "
        "indicating overspending."
    )
else:
    print(
        f"   The savings rate is {savings_rate:.2f}%, "
        "indicating a positive net balance."
    )

print("\n" + "=" * 65)


                 SpendDNA - KEY INSIGHTS

1. SPENDING HABIT
   E-commerce is the highest spending category, accounting for 35.31% of total debits (Rs. 592,816.00).

2. TIME-OF-DAY BEHAVIOUR
   20.64% of Food Delivery transactions happen between 9 PM and 2 AM.

3. SAVINGS BEHAVIOUR
   Total debits exceed total credits. The savings rate is -229.34%, indicating overspending.



In [115]:
# Final Project Validation
print("SpendDNA - FINAL VALIDATION")
print("=" * 60)

checks = {
    "Cleaned dataset available": "df_clean" in globals(),
    "Vendor extraction completed": "vendor_clean" in df_clean.columns,
    "Category tagging completed": "category" in df_clean.columns,
    "Anomaly detection completed": "anomalies" in globals(),
    "Archetype detection completed": "archetypes" in globals(),
    "Monthly trend completed": "monthly_report" in globals(),
    "Time analysis completed": "hour" in debit_data.columns,
    "Final report variables available": "total_credit" in globals()
}

for check, status in checks.items():
    result = "PASS" if status else "CHECK"
    print(f"{check:<38} : {result}")

print("=" * 60)

if all(checks.values()):
    print("All final validation checks passed!")
else:
    print("Please review the checks marked CHECK.")

SpendDNA - FINAL VALIDATION
Cleaned dataset available              : PASS
Vendor extraction completed            : PASS
Category tagging completed             : PASS
Anomaly detection completed            : PASS
Archetype detection completed          : PASS
Monthly trend completed                : PASS
Time analysis completed                : PASS
Final report variables available       : PASS
All final validation checks passed!


In [120]:
import html as html_lib
from IPython.display import display, HTML

dashboard_data = df_clean[
    df_clean["type"] == "debit"
].copy()

dashboard_total_debit = dashboard_data["amount"].sum()

dashboard_total_credit = df_clean[
    df_clean["type"] == "credit"
]["amount"].sum()

dashboard_net = dashboard_total_credit - dashboard_total_debit

dashboard_savings = (
    dashboard_net / dashboard_total_credit * 100
    if dashboard_total_credit > 0 else 0
)

#CATEGORY ANALYSIS

dashboard_categories = (
    dashboard_data.groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

dashboard_category_percent = (
    dashboard_categories / dashboard_total_debit * 100
    if dashboard_total_debit > 0
    else dashboard_categories * 0
)

#TOP VENDORS

dashboard_vendors = (
    dashboard_data.groupby("vendor_clean")["amount"]
    .agg(["sum", "count"])
    .sort_values("sum", ascending=False)
    .head(5)
)

#MONTHLY FOOD DELIVERY

dashboard_food = dashboard_data[
    dashboard_data["category"] == "Food Delivery"
]

dashboard_monthly = dashboard_food.groupby(
    dashboard_food["date"].dt.month
)["amount"].sum()

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

# 5. ANOMALIES AND ARCHETYPES

dashboard_anomalies = anomalies.head(5)

dashboard_archetypes = archetypes

# HELPER FUNCTIONS

def safe_text(value):
    return html_lib.escape(str(value))

def money(value):
    return "₹{:,.0f}".format(value)

# CATEGORY HTML

category_html = ""

for category, amount in dashboard_categories.head(6).items():

    percent = dashboard_category_percent.get(category, 0)

    category_html += f"""
    <div class="category-item">
        <div class="category-top">
            <span>{safe_text(category)}</span>
            <strong>{percent:.1f}%</strong>
        </div>
        <div class="bar-bg">
            <div class="bar-fill"
                 style="width:{min(percent, 100):.2f}%">
            </div>
        </div>
        <div class="amount-small">{money(amount)}</div>
    </div>
    """

# VENDOR HTML

vendor_html = ""

for vendor, row in dashboard_vendors.iterrows():

    vendor_html += f"""
    <tr>
        <td>{safe_text(vendor)}</td>
        <td>{money(row['sum'])}</td>
        <td>{int(row['count'])}</td>
    </tr>
    """

# MONTHLY HTML

monthly_html = ""

if len(dashboard_monthly) > 0:

    max_monthly = dashboard_monthly.max()

    for month, amount in dashboard_monthly.items():

        bar_width = (
            amount / max_monthly * 100
            if max_monthly > 0 else 0
        )

        monthly_html += f"""
        <div class="month-item">
            <div class="month-label">
                <span>{month_names[int(month)-1]}</span>
                <strong>{money(amount)}</strong>
            </div>
            <div class="bar-bg">
                <div class="month-fill"
                     style="width:{bar_width:.2f}%">
                </div>
            </div>
        </div>
        """

else:
    monthly_html = "<p>No monthly food delivery data available.</p>"

# ARCHETYPE HTML

archetype_html = ""

for name, reason in dashboard_archetypes:

    archetype_html += f"""
    <div class="archetype">
        <div class="archetype-name">
            ✦ {safe_text(name)}
        </div>
        <div class="archetype-reason">
            {safe_text(reason)}
        </div>
    </div>
    """

if not archetype_html:
    archetype_html = "<p>No archetypes detected.</p>"

#ANOMALY HTML

anomaly_html = ""

for _, row in dashboard_anomalies.iterrows():

    anomaly_html += f"""
    <tr>
        <td>{safe_text(row['date'].strftime('%d %b %Y'))}</td>
        <td>{safe_text(row['vendor_clean'])}</td>
        <td>{safe_text(row['category'])}</td>
        <td>{money(row['amount'])}</td>
        <td>
            <span class="z-score">{row['z_score']:.2f}</span>
        </td>
    </tr>
    """

if not anomaly_html:
    anomaly_html = """
    <tr>
        <td colspan="5">No unusual transactions detected.</td>
    </tr>
    """

# FINANCIAL INSIGHTS

if len(dashboard_categories) > 0:

    highest_category = dashboard_categories.idxmax()

    highest_percent = dashboard_category_percent.get(
        highest_category, 0
    )

    insight_one = (
        f"{highest_category} is the highest spending category, "
        f"representing {highest_percent:.1f}% of debit spending."
    )

else:
    insight_one = "No category spending information is available."

insight_two = (
    f"{late_night_food_percent:.1f}% of Food Delivery transactions "
    "happen between 9 PM and 2 AM."
)

if dashboard_savings < 0:

    insight_three = (
        f"Debits exceed credits. The savings rate is "
        f"{dashboard_savings:.1f}%."
    )

else:

    insight_three = (
        f"The savings rate is {dashboard_savings:.1f}%, "
        "showing a positive net balance."
    )
dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>

<meta charset="UTF-8">

<style>

* {{
    box-sizing: border-box;
}}

html, body {{
    margin: 0;
    padding: 0;
    width: 1500px;
    height: 900px;
    overflow: hidden;
    background: #121212;
    font-family: Arial, sans-serif;
    color: #FFFFFF;
}}

.dashboard {{
    width: 1500px;
    height: 900px;
    padding: 16px 22px;
    margin: 0;
    background: #121212;
    overflow: hidden;
}}

/* HEADER */

.header {{
    height: 95px;
    background: linear-gradient(120deg, #191919, #242424);
    padding: 15px 22px;
    border-radius: 12px;
    margin-bottom: 10px;
    border: 1px solid #333333;
}}

.header h1 {{
    margin: 0;
    font-size: 27px;
    color: #1DB954;
}}

.header p {{
    margin: 5px 0;
    color: #B3B3B3;
    font-size: 12px;
}}

.header-tag {{
    display: inline-block;
    background: #1DB954;
    color: #000000;
    padding: 5px 10px;
    border-radius: 20px;
    margin-top: 4px;
    font-size: 9px;
    font-weight: bold;
}}

/* TITLES */

.section-title {{
    font-size: 14px;
    font-weight: bold;
    margin: 9px 0 6px;
    color: #FFFFFF;
}}

/* CARDS */

.stats {{
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 10px;
}}

.stat-card {{
    height: 69px;
    background: #181818;
    padding: 10px 14px;
    border-radius: 10px;
    border: 1px solid #303030;
}}

.stat-label {{
    font-size: 10px;
    color: #B3B3B3;
}}

.stat-value {{
    font-size: 19px;
    font-weight: bold;
    margin-top: 6px;
}}

.purple, .green, .blue {{
    color: #1DB954;
}}

.red {{
    color: #FF6B6B;
}}

/* TWO COLUMN LAYOUT */

.two-columns {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 12px;
}}

.panel {{
    background: #181818;
    padding: 11px 14px;
    border-radius: 11px;
    border: 1px solid #303030;
    min-width: 0;
}}

.panel h3 {{
    font-size: 12px;
    margin: 0 0 9px;
    color: #FFFFFF;
}}

/* CATEGORY BARS */

.category-item {{
    margin-bottom: 8px;
}}

.category-top, .month-label {{
    display: flex;
    justify-content: space-between;
    font-size: 10px;
    margin-bottom: 4px;
    color: #E0E0E0;
}}

.bar-bg {{
    height: 6px;
    background: #333333;
    border-radius: 10px;
    overflow: hidden;
}}

.bar-fill {{
    height: 100%;
    background: #1DB954;
    border-radius: 10px;
}}

.month-fill {{
    height: 100%;
    background: #1DB954;
    border-radius: 10px;
}}

.amount-small {{
    font-size: 9px;
    color: #B3B3B3;
    margin-top: 3px;
}}

.month-item {{
    margin-bottom: 9px;
}}

.month-label strong {{
    color: #1DB954;
}}

/* TABLES */

table {{
    width: 100%;
    border-collapse: collapse;
    font-size: 10px;
}}

th {{
    background: #242424;
    color: #1DB954;
    text-align: left;
    padding: 7px;
}}

td {{
    padding: 7px;
    border-bottom: 1px solid #333333;
    color: #E0E0E0;
}}

/* ARCHETYPES */

.archetype-grid {{
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 7px;
}}

.archetype {{
    background: #242424;
    border: 1px solid #333333;
    padding: 8px;
    border-radius: 8px;
}}

.archetype-name {{
    color: #1DB954;
    font-size: 10px;
    font-weight: bold;
}}

.archetype-reason {{
    color: #B3B3B3;
    font-size: 9px;
    margin-top: 4px;
}}

/* ANOMALIES */

.z-score {{
    color: #1DB954;
    background: #193B28;
    padding: 3px 6px;
    border-radius: 5px;
    font-weight: bold;
}}

/* INSIGHTS */

.insights {{
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 10px;
}}

.insight {{
    height: 76px;
    padding: 10px;
    background: #181818;
    border-left: 3px solid #1DB954;
    border-radius: 8px;
    font-size: 10px;
    line-height: 1.4;
    color: #E0E0E0;
    overflow: hidden;
}}

.insight strong {{
    color: #1DB954;
}}

.insight p {{
    margin: 5px 0 0;
}}

/* FOOTER */

.footer {{
    text-align: center;
    color: #777777;
    font-size: 9px;
    margin-top: 8px;
    padding: 5px;
}}

</style>
</head>

<body>

<div class="dashboard">

    <div class="header">

        <h1>✦ SpendDNA</h1>

        <p>
        Discover your spending habits.
        Understand your money.
        Make smarter decisions.
        </p>

        <span class="header-tag">
            PERSONAL FINANCE ANALYTICS | PYTHON + PANDAS
        </span>

    </div>

    <div class="section-title">
        01 / Financial Overview
    </div>

    <div class="stats">

        <div class="stat-card">
            <div class="stat-label">Total Credits</div>
            <div class="stat-value green">
                {money(dashboard_total_credit)}
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-label">Total Debits</div>
            <div class="stat-value purple">
                {money(dashboard_total_debit)}
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-label">Net Change</div>
            <div class="stat-value {'red' if dashboard_net < 0 else 'green'}">
                {money(dashboard_net)}
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-label">Savings Rate</div>
            <div class="stat-value {'red' if dashboard_savings < 0 else 'green'}">
                {dashboard_savings:.1f}%
            </div>
        </div>

    </div>

    <div class="stats" style="margin-top:9px;">

        <div class="stat-card">
            <div class="stat-label">Total Transactions</div>
            <div class="stat-value blue">{len(df_clean):,}</div>
        </div>

        <div class="stat-card">
            <div class="stat-label">Unique Vendors</div>
            <div class="stat-value purple">
                {df_clean['vendor_clean'].nunique()}
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-label">Anomalies Detected</div>
            <div class="stat-value red">{len(anomalies)}</div>
        </div>

        <div class="stat-card">
            <div class="stat-label">Archetypes Found</div>
            <div class="stat-value blue">{len(archetypes)}</div>
        </div>

    </div>

    <div class="two-columns">

        <div>

            <div class="section-title">
                02 / Spending Breakdown
            </div>

            <div class="panel">
                <h3>Top Spending Categories</h3>
                {category_html}
            </div>

        </div>

        <div>

            <div class="section-title">
                03 / Monthly Trends
            </div>

            <div class="panel">

                <h3>Food Delivery Spending</h3>

                {monthly_html}

                <p class="amount-small">
                    Monthly Food Delivery spending.
                </p>

            </div>

        </div>

    </div>

    <div class="two-columns">

        <div>

            <div class="section-title">
                04 / Top Vendors
            </div>

            <div class="panel">

                <h3>Highest Spending Vendors</h3>

                <table>
                    <tr>
                        <th>Vendor</th>
                        <th>Amount</th>
                        <th>Count</th>
                    </tr>

                    {vendor_html}

                </table>

            </div>

        </div>

        <div>

            <div class="section-title">
                05 / Spending Personality
            </div>

            <div class="panel">

                <h3>Your Spending Archetypes</h3>

                <div class="archetype-grid">
                    {archetype_html}
                </div>

            </div>

        </div>

    </div>

    <div class="section-title">
        06 / Unusual Transactions
    </div>

    <div class="panel">

        <h3>Top 5 Anomalies | Category-wise Z-score</h3>

        <table>

            <tr>
                <th>Date</th>
                <th>Vendor</th>
                <th>Category</th>
                <th>Amount</th>
                <th>Z-score</th>
            </tr>

            {anomaly_html}

        </table>

    </div>

    <div class="section-title">
        07 / Financial Insights
    </div>

    <div class="insights">

        <div class="insight">
            <strong>01. Spending Habit</strong>
            <p>{safe_text(insight_one)}</p>
        </div>

        <div class="insight">
            <strong>02. Time Behaviour</strong>
            <p>{safe_text(insight_two)}</p>
        </div>

        <div class="insight">
            <strong>03. Savings Behaviour</strong>
            <p>{safe_text(insight_three)}</p>
        </div>

    </div>

    <div class="footer">

        SpendDNA | Python, Pandas, NumPy, HTML & CSS

        <br>

        Turning transaction data into meaningful financial insights.

    </div>

</div>

</body>
</html>
"""
display(HTML(dashboard_html))
with open(
    "SpendDNA_Dashboard.html",
    "w",
    encoding="utf-8"
) as file:

    file.write(dashboard_html)

print("SpendDNA Dashboard created successfully!")
print("Canvas size: 1500 x 900")
print("Saved as SpendDNA_Dashboard.html")


SpendDNA Dashboard created successfully!
Canvas size: 1500 x 900
Saved as SpendDNA_Dashboard.html
